# 7. Hyperparameters Optimization

## KNN

In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import spearmanr
from chess_eval import *

import numpy as np

NameError: name 'SABED_MODELS_DIR' is not defined

In [ ]:
# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------

dm = load_dataset("one")

In [ ]:
# ----------------------------------------------------------------------
# Unified KNN search
# ----------------------------------------------------------------------

ks = list(range(1, 10)) + list(range(10, 200, 10))

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor()),
])

param_grid = {
    "knn__n_neighbors": ks,
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(dm.X_train, dm.y_train)
best_model = grid.best_estimator_

In [ ]:
# ----------------------------------------------------------------------
# Metric sweep on the SAME grid
# ----------------------------------------------------------------------

X_test_scaled = best_model.named_steps["scaler"].transform(dm.X_test)

spearman_scores = {}
for k in ks:
    model = KNeighborsRegressor(
        n_neighbors=k,
        weights=grid.best_params_["knn__weights"],
        p=grid.best_params_["knn__p"],
        n_jobs=-1
    )

    model.fit(dm.X_train, dm.y_train)
    preds = model.predict(X_test_scaled)
    corr, _ = spearmanr(dm.y_test, preds)
    spearman_scores[k] = corr

    plt.scatter(dm.y_test, preds, alpha=0.5)
    plt.xlim(-EVAL_THRESHOLD, EVAL_THRESHOLD)
    plt.ylim(-EVAL_THRESHOLD, EVAL_THRESHOLD)
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.title("Evaluation Predictions")
    plt.savefig(f"scatter_KNN_sep_{k}.png")
    plt.show()

In [ ]:
# ----------------------------------------------------------------------
# Export for plotting
# ----------------------------------------------------------------------

ks_arr = np.array(list(spearman_scores.keys()))
scores_arr = np.array(list(spearman_scores.values()))

plt.figure(figsize=(12, 6))
plt.plot(ks_arr, scores_arr, marker="o")
plt.xlabel("n_neighbors")
plt.ylabel("Spearman rank correlation")
plt.title("Spearman correlation vs n_neighbors")
plt.grid(True)
plt.show()